In [1]:
# Test data repair scripts 

import sys
import os
from dotenv import load_dotenv, find_dotenv
import matplotlib.pyplot as plt
import time
import pandas as pd
import numpy as np
import json

load_dotenv(find_dotenv())

ROOT_PATH = os.getenv("ROOT_PATH")
MY_DATA_PATH = os.getenv("MY_DATA_PATH")
RAW_DATA_PATH = os.getenv("RAW_DATA_PATH")
DEWEY_PATH = os.path.join(RAW_DATA_PATH, "dewey-downloads", "building-permits-united-states")

sys.path.append(os.path.join(ROOT_PATH, "scripts"))
import data_utils as du

sys.path.append(os.path.join(ROOT_PATH, "agent/scripts"))

from data_repair_ca_santa_monica import data_repair
MY_JURISDICTION = "Santa Monica"

INPUT_FILEPATH = os.path.join(MY_DATA_PATH, "processed_data", "permits_la_sample.parquet")


In [2]:
df = pd.read_parquet(INPUT_FILEPATH)
sub_df = df[df["JURISDICTION"] == MY_JURISDICTION]

for col in ['FILE_DATE', 'PERMIT_DATE', 'FINAL_DATE']:
    sub_df[f'{col}_FLAG'] = ""

#sub_df_filled = sub_df.copy()
sub_df_filled = data_repair(sub_df)

assert(len(sub_df) == len(sub_df_filled))


In [3]:
print(f"FILE_DATE available (all): {sub_df['FILE_DATE'].notna().mean():.1%} -> {sub_df_filled['FILE_DATE'].notna().mean():.1%}")

print(f"PERMIT_DATE available (all): {sub_df['PERMIT_DATE'].notna().mean():.1%} -> {sub_df_filled['PERMIT_DATE'].notna().mean():.1%}")

print(f"FINAL_DATE available (all): {sub_df['FINAL_DATE'].notna().mean():.1%} -> {sub_df_filled['FINAL_DATE'].notna().mean():.1%}")

mask1 = sub_df['STATUS_NORMALIZED'].isin(['Active', 'Final'])
mask2 = sub_df_filled['STATUS_NORMALIZED'].isin(['Active', 'Final'])
print(f"PERMIT_DATE available (active/final): {sub_df.loc[mask1]['PERMIT_DATE'].notna().mean():.1%} -> {sub_df_filled.loc[mask2]['PERMIT_DATE'].notna().mean():.1%}")

mask1 = sub_df['STATUS_NORMALIZED'].isin(['Final'])
mask2 = sub_df_filled['STATUS_NORMALIZED'].isin(['Final'])
print(f"FINAL_DATE available (final): {sub_df.loc[mask1]['FINAL_DATE'].notna().mean():.1%} -> {sub_df_filled.loc[mask2]['FINAL_DATE'].notna().mean():.1%}")


FILE_DATE available (all): 100.0% -> 100.0%
PERMIT_DATE available (all): 36.9% -> 39.4%
FINAL_DATE available (all): 25.6% -> 27.2%
PERMIT_DATE available (active/final): 53.6% -> 58.1%
FINAL_DATE available (final): 53.8% -> 59.1%


In [4]:
for col in ['STATUS_NORMALIZED', 'FILE_DATE', 'PERMIT_DATE', 'FINAL_DATE']:
    print(sub_df_filled[f'{col}_FLAG'].value_counts())

STATUS_NORMALIZED_FLAG
FIXED    35
Name: count, dtype: int64
Series([], Name: count, dtype: int64)
PERMIT_DATE_FLAG
FILLED    51
FIXED      2
Name: count, dtype: int64
FINAL_DATE_FLAG
FILLED    38
FIXED     17
Name: count, dtype: int64


In [5]:
print(sub_df['STATUS_NORMALIZED'].value_counts())
print(sub_df_filled['STATUS_NORMALIZED'].value_counts())

STATUS_NORMALIZED
Final        939
Inactive     567
In Review    335
Active       142
Name: count, dtype: int64
STATUS_NORMALIZED
Final        919
Inactive     570
In Review    330
Active       164
Name: count, dtype: int64


In [6]:
mask = sub_df_filled["FINAL_DATE"].isna()
#mask = sub_df_filled["JURISDICTION"].notna()
sample = sub_df_filled.loc[mask].sample(1).iloc[0]
DATA = sample["DATA"]
DATES_DATA = du.extract_date_fields(DATA) 

print(f"STATUS_NORMALIZED: {sample['STATUS_NORMALIZED']}    *Filled: {sample['STATUS_NORMALIZED_FLAG']}*")
print(f"RECORD_TYPE_ORIGINAL: {sample['RECORD_TYPE_ORIGINAL']}")
print(f"FILE_DATE: {sample['FILE_DATE']}       *Filled: {sample['FILE_DATE_FLAG']}*")
print(f"PERMIT_DATE: {sample['PERMIT_DATE']}   *Filled: {sample['PERMIT_DATE_FLAG']}*")
print(f"FINAL_DATE: {sample['FINAL_DATE']}     *Filled: {sample['FINAL_DATE_FLAG']}*")

print("DATES_DATA: ")
print(json.dumps(DATES_DATA, indent=2))



STATUS_NORMALIZED: Final    *Filled: nan*
RECORD_TYPE_ORIGINAL: Commercial Building Permit
FILE_DATE: 2012-12-07       *Filled: nan*
PERMIT_DATE: None   *Filled: nan*
FINAL_DATE: None     *Filled: nan*
DATES_DATA: 
{
  "date": "2012-12-07",
  "status": "FINAL",
  "search_data": {
    "Date": "12/07/2012",
    "Action": "",
    "Status": "FINAL"
  },
  "fees_details": [
    {
      "Date": "12/07/2012"
    },
    {
      "Date": "12/07/2012"
    },
    {
      "Date": "12/07/2012"
    },
    {
      "Date": "12/07/2012"
    }
  ]
}


In [7]:
print("DATA:")
print(json.dumps(json.loads(DATA), indent=2))



DATA:
{
  "date": "2012-12-07",
  "tasks": [
    {
      "name": "Application Submittal",
      "events": []
    },
    {
      "name": "BS-Arc and Struct Review",
      "events": []
    },
    {
      "name": "BS-Elec Review",
      "events": []
    },
    {
      "name": "BS-Geo Tech Review",
      "events": []
    },
    {
      "name": "BS-Mech Review",
      "events": []
    },
    {
      "name": "BS-Plumb Review",
      "events": []
    },
    {
      "name": "Fire Review",
      "events": []
    },
    {
      "name": "OSE Review",
      "events": []
    },
    {
      "name": "Planning Review",
      "events": []
    },
    {
      "name": "Urban Forestry Review",
      "events": []
    },
    {
      "name": "PW-Civil Eng Review",
      "events": []
    },
    {
      "name": "PW-C and D Review",
      "events": []
    },
    {
      "name": "PW-RRR Review",
      "events": []
    },
    {
      "name": "Rent Control",
      "events": []
    },
    {
      "name": "Transporta